# v0.54 Real CoinEx Execution — Colab

This notebook performs the full empirical v0.54 pipeline: checkout the frozen research branch, install dependencies, run focused and full tests, fetch real CoinEx 1h/4h data, freeze engineered feature frames, run walk-forward ALL/ONLY/DROP audits with 0/24/36 bps costs, replay the frozen snapshot, evaluate the terminal completion gate, and persist all artifacts to Google Drive.

Safety contract: `PAPER_EXECUTION=false`, `LIVE_EXECUTION=false`, Kraken sealed. No order submission occurs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/Colab Notebooks/v54_real_execution')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('Artifacts:', DRIVE_OUT)

In [ ]:
import os, shutil, subprocess, json, sys
REPO = '/content/modular-crypto-trading-bot'
BRANCH = 'research/v54-feature-audit-oos'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/parsa314/modular-crypto-trading-bot.git',REPO], check=True)
os.chdir(REPO)
SHA = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
os.environ['SOURCE_COMMIT'] = SHA
print('Checked out:', SHA)
assert subprocess.check_output(['git','branch','--show-current'], text=True).strip() == BRANCH

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--upgrade','pip'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[dev]'], check=True)
subprocess.run([sys.executable,'-m','compileall','-q','research_bot','scripts','tests'], check=True)
print('Install + compile: PASS')

In [ ]:
focused = [
 'tests/test_v53_candle_zone_mtf.py',
 'tests/test_v53_confluence_forward.py',
 'tests/test_v53_confluence_missing.py',
 'tests/test_feature_audit_v54.py',
 'tests/test_v54_integrity.py',
 'tests/test_v54_completion.py',
]
subprocess.run([sys.executable,'-m','pytest','-q',*focused], check=True)
print('Focused v0.53/v0.54 tests: PASS')

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)
print('Full repository suite: PASS')

In [ ]:
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOCAL = Path('/content/v54_run') / RUN_ID
FREEZE = LOCAL / 'frozen'
LOCAL.mkdir(parents=True, exist_ok=True)
cmd = [
 sys.executable, 'scripts/run_v54_feature_audit.py',
 '--symbols','BTC/USDT','ETH/USDT','SOL/USDT','XRP/USDT','DOGE/USDT',
 '--one-hour-bars','3000', '--four-hour-bars','1200',
 '--min-train','1200', '--test-rows','240', '--step-rows','240',
 '--bootstrap-resamples','2000', '--bootstrap-block','24',
 '--freeze-dir', str(FREEZE),
 '--output', str(LOCAL / 'feature_audit.json')
]
print('Running fresh CoinEx audit...')
subprocess.run(cmd, check=True)
print('Fresh empirical run complete:', LOCAL / 'feature_audit.json')

In [ ]:
replay_cmd = [
 sys.executable, 'scripts/run_v54_feature_audit.py',
 '--symbols','BTC/USDT','ETH/USDT','SOL/USDT','XRP/USDT','DOGE/USDT',
 '--min-train','1200', '--test-rows','240', '--step-rows','240',
 '--bootstrap-resamples','2000', '--bootstrap-block','24',
 '--replay-dir', str(FREEZE),
 '--output', str(LOCAL / 'feature_audit_replay.json')
]
print('Running exact frozen replay...')
subprocess.run(replay_cmd, check=True)
print('Replay complete')

In [ ]:
from research_bot.v54_completion import evaluate_v54_completion
fresh = json.loads((LOCAL/'feature_audit.json').read_text())
replay = json.loads((LOCAL/'feature_audit_replay.json').read_text())
gate = evaluate_v54_completion(fresh)
(LOCAL/'completion_gate.json').write_text(json.dumps(gate, indent=2, sort_keys=True)+'\n')
print(json.dumps(gate, indent=2))
assert fresh['source_commit'] == SHA, (fresh['source_commit'], SHA)
assert fresh['dataset_manifests'].keys() == replay['dataset_manifests'].keys()
for s in fresh['dataset_manifests']:
    assert fresh['dataset_manifests'][s]['frame_sha256'] == replay['dataset_manifests'][s]['frame_sha256']
    assert fresh['dataset_manifests'][s]['schema_sha256'] == replay['dataset_manifests'][s]['schema_sha256']
print('Fresh/replay manifest equality: PASS')

In [ ]:
print('STATUS:', gate['status'])
print('Completed symbols:', fresh.get('symbols_completed'))
print('Blocked:', json.dumps(fresh.get('blocked', {}), indent=2))
print('\nFamily evidence:')
families = fresh.get('cross_symbol_evidence',{}).get('families',{})
for name, row in families.items():
    print(name, 'promotion=', row.get('promotion_candidate'), 'paired_support=', row.get('paired_return_support'), 'median_delta_sharpe=', row.get('median_all_minus_drop_sharpe'))
assert fresh.get('paper_execution') is False
assert fresh.get('live_execution') is False


In [ ]:
DEST = DRIVE_OUT / RUN_ID
if DEST.exists(): shutil.rmtree(DEST)
shutil.copytree(LOCAL, DEST)
archive = shutil.make_archive(str(DRIVE_OUT / f'v54_{RUN_ID}'), 'zip', LOCAL)
print('Saved artifacts to Drive:', DEST)
print('ZIP:', archive)
print('SOURCE_COMMIT:', SHA)